# Grad-CAM Interpretability — VIA-DIA

Loads a trained checkpoint and visualises which spatial regions of the pre/post satellite images drive the model's damage-class prediction.

**Supported architectures:** `BaselineMultimodalModel`, `ImageDifferenceWeatherGRUModel`, `ImageDifferenceCNNModel`  
**Not supported:** `WeatherGRUModel` (no visual backbone)

## 0 · Setup

In [ ]:
import os, sys

# Make sure we run from the project root so relative imports work
PROJECT_ROOT = os.path.abspath('.')          # adjust if the notebook is not in the repo root
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import yaml
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')   # keeps lightning_module happy; we override per cell below
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

from lightning_module import DIA
from models.grad_cam import visualize_grad_cam, get_target_layer, MultimodalGradCAM

## 1 · Configure paths

Edit the three variables below, then run the rest of the notebook.

In [ ]:
# ── Required ──────────────────────────────────────────────────────────────────
CHECKPOINT_PATH = "./experiments/path/to/your/checkpoint.ckpt"   # Lightning .ckpt file
ARCH            = "BaselineMultimodalModel"                       # model architecture name
CACHE_DIR       = "/mnt/database/xBD"                            # preprocessed xBD cache

# ── Optional ──────────────────────────────────────────────────────────────────
CONFIG_FILE     = "./configs/config_baseline.yaml"  # baseline config (overridden below)
OUTPUT_DIR      = "./gradcam_results"               # where figures are saved
TARGET_CLASS    = None   # int (0-3) to filter samples, or None for all classes
MAX_SAMPLES     = 10     # total figures to generate
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")
print(f"Checkpoint exists: {os.path.isfile(CHECKPOINT_PATH)}")

## 2 · Build config and instantiate model

In [ ]:
with open(CONFIG_FILE) as f:
    cfg = yaml.safe_load(f)

# Override fields so we don't need a separate interpretation config
cfg['data']['cache_dir']    = CACHE_DIR
cfg['data']['batch_size']   = 16
cfg['data']['augment']      = False
cfg['trainer']['save_dir']  = OUTPUT_DIR
cfg['arch']['initial_weights'] = ""   # we load via state_dict below

os.makedirs(OUTPUT_DIR, exist_ok=True)

dia = DIA(cfg, ARCH, database='xBDClimate')
print(f"Model: {type(dia.model).__name__}")

## 3 · Load checkpoint weights

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
missing, unexpected = dia.load_state_dict(ckpt['state_dict'], strict=False)

if missing:
    print(f"Missing keys ({len(missing)}): {missing[:5]} ...")
if unexpected:
    print(f"Unexpected keys ({len(unexpected)}): {unexpected[:5]} ...")

dia.model.to(DEVICE)
dia.model.eval()
print("Checkpoint loaded.")

## 4 · Generate Grad-CAM figures (batch mode)

Iterates the test set, saves one PNG per sample to `OUTPUT_DIR`.

In [ ]:
dia.grad_cam_interpretation(
    device=DEVICE,
    target_cat=TARGET_CLASS,
    max_samples=MAX_SAMPLES,
)
print("Done — figures saved to", OUTPUT_DIR)

## 5 · Display saved figures inline

In [ ]:
saved = sorted(
    [os.path.join(OUTPUT_DIR, f) for f in os.listdir(OUTPUT_DIR) if f.endswith('_gradcam.png')]
)
print(f"Found {len(saved)} Grad-CAM figures.")

for path in saved:
    print(os.path.basename(path))
    display(IPImage(filename=path))

## 6 · Single-sample deep-dive (inline, no file save)

Pick one sample from the test loader and inspect Grad-CAM interactively.

In [ ]:
# Grab the first batch
batch = next(iter(dia.data_test))
(patches_pre, patches_post, mask_patches, climate_series,
 event_labels, labels_pre, labels_post, event_names, patch_idxs) = batch

SAMPLE_IDX = 0   # change to inspect a different sample in the batch

x_pre     = patches_pre[[SAMPLE_IDX]].to(DEVICE)
x_post    = patches_post[[SAMPLE_IDX]].to(DEVICE)
x_climate = climate_series[[SAMPLE_IDX]].to(DEVICE)
ev_labels = event_labels[[SAMPLE_IDX]].to(DEVICE) if event_labels is not None else None
gt_label  = int(labels_post[SAMPLE_IDX].item())

dataset       = dia.data_test.dataset
damage_classes = getattr(dataset, 'damage_classes', None)
patch_mean    = getattr(dataset, 'patch_mean', None)
patch_std     = getattr(dataset, 'patch_std', None)
if patch_mean is not None:
    patch_mean = patch_mean.numpy() if hasattr(patch_mean, 'numpy') else np.asarray(patch_mean)
if patch_std is not None:
    patch_std = patch_std.numpy() if hasattr(patch_std, 'numpy') else np.asarray(patch_std)

name = event_names[SAMPLE_IDX].split('/')[-1][:-3] if isinstance(event_names[SAMPLE_IDX], str) else str(SAMPLE_IDX)
print(f"Sample : {name}")
print(f"GT     : {list(damage_classes.keys())[gt_label] if damage_classes else gt_label}")

In [ ]:
# Run Grad-CAM and render inline
import cv2
import torch.nn.functional as F

matplotlib.use('inline' if 'inline' in matplotlib.get_backend() else 'Agg')
%matplotlib inline

target_layer = get_target_layer(dia.model)
grad_cam     = MultimodalGradCAM(dia.model, target_layer)

cam_post, att_weights, pred_class = grad_cam.generate(
    x_pre, x_post, x_climate, ev_labels, target_image='post'
)
cam_pre, _, _ = grad_cam.generate(
    x_pre, x_post, x_climate, ev_labels,
    target_class=pred_class, target_image='pre'
)
grad_cam.remove_hooks()

def to_rgb(tensor, mean=None, std=None):
    img = tensor[0].detach().permute(1, 2, 0).cpu().numpy()
    if mean is not None and std is not None:
        img = (img * 3.0 * std.reshape(1,1,-1) + mean.reshape(1,1,-1)).clip(0,255).astype(np.uint8) / 255.
    else:
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img.astype(np.float32)

img_pre  = to_rgb(x_pre,  patch_mean, patch_std)
img_post = to_rgb(x_post, patch_mean, patch_std)

cam_pre_r  = cv2.resize(cam_pre,  (img_pre.shape[1],  img_pre.shape[0]))
cam_post_r = cv2.resize(cam_post, (img_post.shape[1], img_post.shape[0]))

has_att = len(att_weights) > 0
ncols   = 3 if has_att else 2
fig, axes = plt.subplots(1, ncols, figsize=(5*ncols, 5))

axes[0].imshow(img_pre)
axes[0].imshow(cam_pre_r,  cmap='jet', alpha=0.45)
axes[0].set_title('Pre-Disaster Grad-CAM')
axes[0].axis('off')

axes[1].imshow(img_post)
axes[1].imshow(cam_post_r, cmap='jet', alpha=0.45)
axes[1].set_title('Post-Disaster Grad-CAM')
axes[1].axis('off')

if has_att:
    axes[2].bar(np.arange(len(att_weights)), att_weights, color='teal')
    axes[2].set_title('Climate Temporal Attention')
    axes[2].set_xlabel('Time Step')
    axes[2].set_ylabel('Attention Weight')

class_names = list(damage_classes.keys()) if damage_classes else None
pred_name   = class_names[pred_class] if class_names else str(pred_class)
gt_name     = class_names[gt_label]   if class_names else str(gt_label)
fig.suptitle(f"{name} — GT: {gt_name}  Pred: {pred_name}", fontsize=13)
plt.tight_layout()
plt.show()

## 7 · Per-class heatmap comparison

Show the Grad-CAM for every possible target class on the same post-image to see how the saliency shifts.

In [ ]:
num_classes  = cfg['arch']['num_classes']
class_names_ = list(damage_classes.keys()) if damage_classes else [str(i) for i in range(num_classes)]

fig, axes = plt.subplots(1, num_classes, figsize=(5*num_classes, 5))

for cls_idx in range(num_classes):
    target_layer_ = get_target_layer(dia.model)
    gc = MultimodalGradCAM(dia.model, target_layer_)
    cam, _, _ = gc.generate(
        x_pre, x_post, x_climate, ev_labels,
        target_class=cls_idx, target_image='post'
    )
    gc.remove_hooks()

    cam_r = cv2.resize(cam, (img_post.shape[1], img_post.shape[0]))
    axes[cls_idx].imshow(img_post)
    axes[cls_idx].imshow(cam_r, cmap='jet', alpha=0.45)
    axes[cls_idx].set_title(f'Class {cls_idx}: {class_names_[cls_idx]}')
    axes[cls_idx].axis('off')

fig.suptitle(f"Post-image Grad-CAM — one heatmap per target class\n{name}", fontsize=13)
plt.tight_layout()
plt.show()